# Clasificacion de Pokemon segun su tipo con InceptionV3

Proyecto final de **Introduccion a la Inteligencia Artificial** - Licenciatura en Ciencias de la Computacion, UNR.

**Integrantes:** Ignacio Basualdo, Lautaro Capezio y Luciano Duarte.

El objetivo de este trabajo es adaptar un modelo convolucional preentrenado, **InceptionV3**, a una tarea de clasificacion de imagenes de Pokemon. El modelo recibe como entrada la imagen de un Pokemon y predice su tipo primario (`Type 1`) entre 18 clases posibles. La eleccion sigue la opcion de fine-tuning con InceptionV3 del enunciado del TP final, y se apoya en los temas vistos en la materia: redes neuronales, redes convolucionales profundas, transfer learning, regularizacion, evaluacion y analisis de resultados.

La version actual del notebook busca dejar el proyecto en un estado defendible: carga y analisis del dataset, justificacion metodologica, implementacion reproducible, registro de corridas, metricas, graficas y discusion final.

## 1. Planteo del problema

El problema se plantea como una clasificacion multiclase supervisada. Cada ejemplo del dataset contiene una imagen y metadatos del Pokemon; en este trabajo usamos la imagen como entrada y el campo `Type 1` como etiqueta. La variable objetivo tiene 18 valores posibles: `normal`, `fire`, `water`, `grass`, `electric`, `ice`, `fighting`, `poison`, `ground`, `flying`, `psychic`, `bug`, `rock`, `ghost`, `dragon`, `steel`, `fairy` y `dark`.

Desde el punto de vista de Inteligencia Artificial, la tarea es interesante porque no todos los tipos se distinguen de manera puramente visual. Algunas clases tienen senales bastante claras, por ejemplo fuego, agua o planta; otras dependen mas del diseno del personaje, la generacion, el color o convenciones internas de la franquicia. Por eso esperamos que el modelo aprenda regularidades visuales utiles, pero tambien que tenga confusiones razonables entre tipos esteticamente cercanos.

## 2. Relacion con la teoria y con el enunciado

InceptionV3 es una red convolucional profunda pensada originalmente para clasificacion de imagenes. Al estar preentrenada sobre ImageNet, sus primeras y medias capas ya aprendieron detectores visuales generales: bordes, texturas, patrones locales, formas y composiciones mas complejas. La estrategia del TP consiste en transferir esa representacion a nuestro problema particular.

En lugar de entrenar toda la red desde cero, reemplazamos la capa final por una nueva capa con 18 salidas y hacemos fine-tuning sobre una parte reducida del modelo. Esta decision reduce el costo computacional y ayuda cuando el dataset propio no es enorme. A la vez, descongelar algunas capas profundas permite que el extractor se adapte al dominio Pokemon, que no coincide exactamente con fotografias naturales de ImageNet.

In [ ]:
# Si se ejecuta en Colab o en un entorno limpio, puede hacer falta instalar dependencias:
# !pip install -q datasets torch torchvision scikit-learn seaborn

from pathlib import Path
import json
import os
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

from datasets import load_dataset
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Si hay GPU, PyTorch usa CUDA. Si no, el notebook igual funciona en CPU, aunque bastante mas lento.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Dataset

Usamos el dataset `cnxiaomai/pokemon-classification-gen1-9` disponible en Hugging Face. Contiene Pokemon de generaciones 1 a 9 con imagenes y campos descriptivos. Para esta primera version, el target elegido es `Type 1`, es decir, el tipo principal. Dejamos afuera `Type 2` porque convertiria el problema en multi-label y exigira otra funcion de perdida, otra interpretacion de metricas y otra discusion; es una extension posible, pero no la primera version aprobada en la propuesta.

El analisis del dataset sirve para revisar tres cuestiones importantes:

- Que campos estan disponibles.
- Si la distribucion de clases esta balanceada o no.
- Como se ven las imagenes antes de pasar por el modelo.

In [ ]:
print("Cargando dataset desde Hugging Face...")
hf_dataset = load_dataset("cnxiaomai/pokemon-classification-gen1-9")
print(hf_dataset)

train_raw = hf_dataset["train"]
test_raw = hf_dataset["test"]

print(f"Cantidad de ejemplos de entrenamiento: {len(train_raw)}")
print(f"Cantidad de ejemplos de test/validacion: {len(test_raw)}")
print(f"Columnas disponibles: {train_raw.column_names}")

In [ ]:
POKEMON_TYPES = [
    "normal", "fire", "water", "grass", "electric", "ice",
    "fighting", "poison", "ground", "flying", "psychic", "bug",
    "rock", "ghost", "dragon", "steel", "fairy", "dark"
]

type_to_idx = {t: i for i, t in enumerate(POKEMON_TYPES)}
idx_to_type = {i: t for t, i in type_to_idx.items()}
num_types = len(POKEMON_TYPES)


def normalize_type(value):
    return str(value).lower().strip()


def dataset_to_frame(split):
    rows = []
    for item in split:
        rows.append({
            "name": item.get("Name", item.get("name", None)),
            "type_1": normalize_type(item["Type 1"]),
            "type_2": normalize_type(item.get("Type 2", "")) if item.get("Type 2", None) is not None else "",
        })
    return pd.DataFrame(rows)

train_df = dataset_to_frame(train_raw)
test_df = dataset_to_frame(test_raw)

display(train_df.head())
print("Tipos encontrados en train:", sorted(train_df["type_1"].unique()))
print("Tipos esperados que no aparecen en train:", sorted(set(POKEMON_TYPES) - set(train_df["type_1"].unique())))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

train_counts = train_df["type_1"].value_counts().reindex(POKEMON_TYPES, fill_value=0)
test_counts = test_df["type_1"].value_counts().reindex(POKEMON_TYPES, fill_value=0)

train_counts.plot(kind="bar", ax=ax[0], color="#4C78A8")
ax[0].set_title("Distribucin de clases - train")
ax[0].set_xlabel("Tipo primario")
ax[0].set_ylabel("Cantidad de imagenes")
ax[0].tick_params(axis="x", rotation=45)

test_counts.plot(kind="bar", ax=ax[1], color="#F58518")
ax[1].set_title("Distribucin de clases - test")
ax[1].set_xlabel("Tipo primario")
ax[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

class_distribution = pd.DataFrame({"train": train_counts, "test": test_counts})
display(class_distribution)
print(f"Clase mayoritaria en train: {train_counts.idxmax()} ({train_counts.max()} ejemplos)")
print(f"Clase minoritaria en train: {train_counts.idxmin()} ({train_counts.min()} ejemplos)")

### Observacion sobre balance de clases

La distribucion por tipo no suele ser uniforme. Esto importa porque una accuracy global puede ocultar que el modelo aprende mejor las clases con mas ejemplos y peor las clases poco representadas. Por ese motivo, ademas de reportar accuracy, conviene mirar matriz de confusion y metricas por clase (`precision`, `recall` y `f1-score`).

In [ ]:
def show_raw_samples(split, n=12):
    n = min(n, len(split))
    fig = plt.figure(figsize=(14, 8))
    for i in range(n):
        item = split[i]
        image = item["image_data"].convert("RGB")
        label = normalize_type(item["Type 1"])
        name = item.get("Name", item.get("name", f"#{i}"))
        plt.subplot(3, 4, i + 1)
        plt.imshow(image)
        plt.title(f"{name}\nTipo: {label}", fontsize=10)
        plt.axis("off")
    plt.suptitle("Muestra de imagenes del dataset", fontsize=16)
    plt.tight_layout()
    plt.show()

show_raw_samples(train_raw, n=12)

## 4. Preprocesamiento

InceptionV3 espera imagenes de tamano `299 x 299` y tres canales de color. Algunas imagenes pueden venir con canal alfa/transparencia, por eso convertimos cada ejemplo a RGB. Para entrenamiento usamos data augmentation moderado: espejado horizontal, rotacion leve y pequenas variaciones de brillo/contraste. La idea es mejorar la generalizacion sin destruir rasgos visuales importantes del Pokemon.

Para validacion y test no aplicamos transformaciones aleatorias, porque queremos medir el desempeno sobre imagenes estables y comparables entre corridas.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class PokemonDataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.hf_data = hf_data
        self.transform = transform
        self.labels = [type_to_idx[normalize_type(item["Type 1"])] for item in hf_data]

    def __len__(self):
        return len(self.hf_data)

    def __getitem__(self, idx):
        item = self.hf_data[idx]
        image = item["image_data"].convert("RGBA").convert("RGB")
        if self.transform:
            image = self.transform(image)

        type_str = normalize_type(item["Type 1"])
        if type_str not in type_to_idx:
            raise ValueError(f"Tipo desconocido: {type_str}")
        label = type_to_idx[type_str]

        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "name": item.get("Name", item.get("name", str(idx))),
        }

train_dataset = PokemonDataset(train_raw, transform=train_transforms)
val_dataset = PokemonDataset(test_raw, transform=val_transforms)

# En Windows, usar workers > 0 desde notebooks puede traer problemas de multiprocessing.
# En Colab/Linux se aprovechan hasta 4 workers.
num_workers = 0 if os.name == "nt" else min(4, os.cpu_count() or 2)
pin_memory = device.type == "cuda"
BATCH_SIZE = 64

class_sample_counts = np.bincount(train_dataset.labels, minlength=num_types)
class_weights_np = len(train_dataset) / (num_types * np.maximum(class_sample_counts, 1))
class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)


def make_train_loader(use_weighted_sampler=False, batch_size=BATCH_SIZE):
    if use_weighted_sampler:
        sample_weights = class_weights_np[train_dataset.labels]
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
        )
        return DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=sampler,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    return DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

train_loader = make_train_loader(use_weighted_sampler=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
print(f"num_workers={num_workers}, pin_memory={pin_memory}")
print("Pesos por clase para loss balanceada:")
display(pd.DataFrame({"type": POKEMON_TYPES, "count": class_sample_counts, "weight": class_weights_np}))

batch = next(iter(train_loader))
print(batch["image"].shape, batch["label"].shape)

In [ ]:
# Visualizacion de algunas imagenes luego de las transformaciones de entrenamiento.
# Para mostrarlas se desnormalizan con la media y el desvio estandar de ImageNet.

def denormalize_image(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = tensor.cpu() * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

batch = next(iter(train_loader))
fig = plt.figure(figsize=(14, 8))
for i in range(min(12, batch["image"].shape[0])):
    plt.subplot(3, 4, i + 1)
    plt.imshow(denormalize_image(batch["image"][i]))
    plt.title(idx_to_type[int(batch["label"][i])], fontsize=10)
    plt.axis("off")
plt.suptitle("Ejemplos transformados para entrenamiento", fontsize=16)
plt.tight_layout()
plt.show()

## 5. Modelo: InceptionV3 + fine-tuning

Partimos de los pesos preentrenados de `torchvision`. Primero congelamos todos los parametros y luego descongelamos solo capas profundas. Esto sigue la intuicin de transfer learning: las primeras capas aprenden patrones muy generales, mientras que las ultimas capas codifican caracteristicas mas especificas de la tarea original.

En nuestra corrida inicial descongelamos `Mixed_7c` y reemplazamos tanto la salida principal (`fc`) como la salida auxiliar (`AuxLogits.fc`). La salida auxiliar es propia de InceptionV3 durante entrenamiento y ayuda a propagar gradientes en redes profundas; por eso, en entrenamiento combinamos la perdida principal con `0.4 * perdida_auxiliar`.

In [ ]:
def make_classifier_head(in_features, num_classes, dropout=0.0):
    if dropout and dropout > 0:
        return nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes),
        )
    return nn.Linear(in_features, num_classes)


def build_inception_model(num_classes=18, fine_tune="mixed_7c", dropout=0.0):
    weights = models.Inception_V3_Weights.DEFAULT
    model = models.inception_v3(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    if fine_tune == "classifier_only":
        trainable_blocks = []
    elif fine_tune == "mixed_7c":
        trainable_blocks = [model.Mixed_7c]
    elif fine_tune == "mixed_7b_7c":
        trainable_blocks = [model.Mixed_7b, model.Mixed_7c]
    elif fine_tune == "mixed_7a_7b_7c":
        trainable_blocks = [model.Mixed_7a, model.Mixed_7b, model.Mixed_7c]
    else:
        raise ValueError(f"Configuracion de fine_tune no soportada: {fine_tune}")

    for block in trainable_blocks:
        for param in block.parameters():
            param.requires_grad = True

    fc_in_features = model.fc.in_features
    aux_in_features = model.AuxLogits.fc.in_features
    model.fc = make_classifier_head(fc_in_features, num_classes, dropout=dropout)
    model.AuxLogits.fc = make_classifier_head(aux_in_features, num_classes, dropout=dropout)
    return model


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def summarize_model_config(fine_tune="mixed_7c", dropout=0.0):
    temp_model = build_inception_model(num_classes=num_types, fine_tune=fine_tune, dropout=dropout)
    total_params, trainable_params = count_parameters(temp_model)
    return {
        "fine_tune": fine_tune,
        "dropout": dropout,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_percent": 100 * trainable_params / total_params,
    }

model = build_inception_model(num_classes=num_types, fine_tune="mixed_7c", dropout=0.0).to(device)
total_params, trainable_params = count_parameters(model)
print(f"Parametros totales: {total_params:,}")
print(f"Parametros entrenables: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

## 6. Optimizacion y entrenamiento

Usamos `CrossEntropyLoss`, adecuada para clasificacion multiclase con una sola etiqueta correcta por imagen. El optimizador elegido es Adam con learning rates diferenciados: una tasa menor para las capas preentrenadas descongeladas y una tasa mayor para las capas finales nuevas. Esto evita modificar de golpe los pesos ya aprendidos y, al mismo tiempo, permite que el clasificador nuevo aprenda rapido.

El scheduler `StepLR` reduce el learning rate cada 5 epocas. En la corrida inicial se entreno durante 15 epocas.

In [ ]:
def make_criterion(use_class_weights=False):
    if use_class_weights:
        return nn.CrossEntropyLoss(weight=class_weights)
    return nn.CrossEntropyLoss()


def make_optimizer(model, fine_tune_lr=1e-4, head_lr=1e-3, weight_decay=0.0, optimizer_name="adam"):
    parameter_groups = []

    # Capas profundas descongeladas.
    deep_params = [p for name, p in model.named_parameters()
                   if p.requires_grad and not name.startswith("fc") and not name.startswith("AuxLogits.fc")]
    if deep_params:
        parameter_groups.append({"params": deep_params, "lr": fine_tune_lr})

    parameter_groups.append({"params": model.fc.parameters(), "lr": head_lr})
    parameter_groups.append({"params": model.AuxLogits.fc.parameters(), "lr": head_lr})

    if optimizer_name.lower() == "adamw":
        return optim.AdamW(parameter_groups, weight_decay=weight_decay)
    if optimizer_name.lower() == "adam":
        return optim.Adam(parameter_groups, weight_decay=weight_decay)
    raise ValueError(f"Optimizador no soportado: {optimizer_name}")

criterion = make_criterion(use_class_weights=False)
optimizer = make_optimizer(model, fine_tune_lr=1e-4, head_lr=1e-3, weight_decay=0.0, optimizer_name="adam")
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(optimizer)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    running_corrects = 0
    total = 0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            if is_train:
                outputs, aux_outputs = model(images)
                loss_main = criterion(outputs, labels)
                loss_aux = criterion(aux_outputs, labels)
                loss = loss_main + 0.4 * loss_aux
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)

            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        running_corrects += torch.sum(preds == labels.data).item()
        total += images.size(0)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())

    return {
        "loss": running_loss / total,
        "accuracy": running_corrects / total,
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler=None,
    epochs=15,
    run_name="inception_mixed_7c",
    early_stopping_patience=None,
):
    history = []
    best_acc = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    best_path = OUTPUT_DIR / f"{run_name}_best.pt"

    for epoch in range(epochs):
        print(f"\nEpoca {epoch + 1}/{epochs}")
        print("-" * 20)

        train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None)

        if scheduler is not None:
            scheduler.step()

        row = {
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        }
        history.append(row)
        print(f"Train Loss: {row['train_loss']:.4f} Acc: {row['train_acc']:.4f} Macro-F1: {row['train_macro_f1']:.4f}")
        print(f"Val   Loss: {row['val_loss']:.4f} Acc: {row['val_acc']:.4f} Macro-F1: {row['val_macro_f1']:.4f}")

        if row["val_acc"] > best_acc:
            best_acc = row["val_acc"]
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "history": history,
                "idx_to_type": idx_to_type,
                "type_to_idx": type_to_idx,
                "run_name": run_name,
                "best_epoch": best_epoch,
                "best_val_acc": best_acc,
            }, best_path)
        else:
            epochs_without_improvement += 1

        if early_stopping_patience is not None and epochs_without_improvement >= early_stopping_patience:
            print(f"Early stopping: sin mejora de val_acc durante {early_stopping_patience} epocas.")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(OUTPUT_DIR / f"{run_name}_history.csv", index=False)
    print(f"Mejor val_acc: {best_acc:.4f} en epoca {best_epoch}")
    print(f"Checkpoint guardado en: {best_path}")
    return history_df, best_path

# Para correr nuevamente el entrenamiento baseline completo, descomentar:
# history_df, best_model_path = train_model(
#     model, train_loader, val_loader, criterion, optimizer, scheduler,
#     epochs=15, run_name="inception_mixed_7c"
# )

## 7. Corrida registrada

La corrida inicial que quedo guardada en el notebook anterior uso la siguiente configuracion:

- Modelo: InceptionV3 preentrenado en ImageNet.
- Fine-tuning: `Mixed_7c` + capa final + salida auxiliar.
- Optimizador: Adam.
- Learning rates: `1e-4` para `Mixed_7c` y `1e-3` para las capas finales.
- Scheduler: `StepLR(step_size=5, gamma=0.5)`.
- epocas: 15.
- Batch size: 64.
- Metrica principal: accuracy sobre validacion/test.

A continuacion se cargan manualmente esas metricas para poder graficarlas y discutirlas aunque no se reentrene el modelo cada vez que se abre el notebook.

In [ ]:
historical_run = pd.DataFrame([
    {"epoch": 1, "train_loss": 3.4197, "train_acc": 0.2514, "val_loss": 2.1750, "val_acc": 0.3368},
    {"epoch": 2, "train_loss": 2.8722, "train_acc": 0.3808, "val_loss": 1.9728, "val_acc": 0.3901},
    {"epoch": 3, "train_loss": 2.5245, "train_acc": 0.4740, "val_loss": 1.8720, "val_acc": 0.4265},
    {"epoch": 4, "train_loss": 2.2560, "train_acc": 0.5538, "val_loss": 1.8097, "val_acc": 0.4478},
    {"epoch": 5, "train_loss": 2.0062, "train_acc": 0.6280, "val_loss": 1.6223, "val_acc": 0.5174},
    {"epoch": 6, "train_loss": 1.7270, "train_acc": 0.7113, "val_loss": 1.5371, "val_acc": 0.5383},
    {"epoch": 7, "train_loss": 1.6011, "train_acc": 0.7504, "val_loss": 1.5871, "val_acc": 0.5383},
    {"epoch": 8, "train_loss": 1.5393, "train_acc": 0.7646, "val_loss": 1.5481, "val_acc": 0.5494},
    {"epoch": 9, "train_loss": 1.4705, "train_acc": 0.7865, "val_loss": 1.5508, "val_acc": 0.5522},
    {"epoch": 10, "train_loss": 1.3839, "train_acc": 0.8114, "val_loss": 1.5302, "val_acc": 0.5597},
    {"epoch": 11, "train_loss": 1.2991, "train_acc": 0.8339, "val_loss": 1.5211, "val_acc": 0.5660},
    {"epoch": 12, "train_loss": 1.2626, "train_acc": 0.8486, "val_loss": 1.5193, "val_acc": 0.5755},
    {"epoch": 13, "train_loss": 1.2493, "train_acc": 0.8513, "val_loss": 1.5491, "val_acc": 0.5779},
    {"epoch": 14, "train_loss": 1.2229, "train_acc": 0.8557, "val_loss": 1.5495, "val_acc": 0.5771},
    {"epoch": 15, "train_loss": 1.1964, "train_acc": 0.8699, "val_loss": 1.5162, "val_acc": 0.5846},
])

display(historical_run)
print(f"Mejor accuracy de validacion: {historical_run['val_acc'].max():.4f}")
print(f"Epoca del mejor accuracy: {int(historical_run.loc[historical_run['val_acc'].idxmax(), 'epoch'])}")

In [ ]:
def plot_history(history_df, title="Entrenamiento InceptionV3"):
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    ax[0].plot(history_df["epoch"], history_df["train_acc"], marker="o", label="Train accuracy")
    ax[0].plot(history_df["epoch"], history_df["val_acc"], marker="o", label="Validation accuracy")
    ax[0].set_title("Accuracy")
    ax[0].set_xlabel("Epoca")
    ax[0].set_ylabel("Accuracy")
    ax[0].set_ylim(0, 1)
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
    ax[1].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
    ax[1].set_title("Loss")
    ax[1].set_xlabel("Epoca")
    ax[1].set_ylabel("Loss")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

plot_history(historical_run, title="Corrida registrada - InceptionV3 con Mixed_7c")

### Lectura de la corrida

La accuracy de entrenamiento crece desde `0.2514` hasta `0.8699`, mientras que la accuracy de validacion pasa de `0.3368` a `0.5846`. Esto muestra que el modelo efectivamente aprende: supera por bastante el azar de 18 clases, que estaria cerca de `1/18 = 0.0556`. Sin embargo, tambien aparece una brecha considerable entre entrenamiento y validacion, sobre todo desde la mitad de la corrida. Esa diferencia sugiere sobreajuste parcial.

La perdida de validacion mejora fuerte hasta aproximadamente la Epoca 6 y luego se estabiliza con pequenas oscilaciones. La accuracy de validacion sigue mejorando lentamente hasta la Epoca 15, pero con ganancias cada vez menores. Por lo tanto, la corrida es util como baseline, aunque todavia hay margen para probar mas regularizacion, otra profundidad de fine-tuning y estrategias de balance de clases.

## 8. Evaluacion detallada

La siguiente seccion sirve para evaluar un modelo entrenado. Si se guardo un checkpoint en `outputs/inception_mixed_7c_best.pt`, se puede cargar y obtener matriz de confusion, reporte por clase y ejemplos cualitativos. Estas herramientas son importantes para el informe porque la accuracy global no alcanza para entender que tipos se confunden entre si.

In [ ]:
def load_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model, checkpoint


def predict_loader(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    names = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            labels = batch["label"].cpu().numpy()
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()

            y_true.extend(labels.tolist())
            y_pred.extend(preds.tolist())
            names.extend(batch.get("name", [""] * len(labels)))

    return np.array(y_true), np.array(y_pred), names


def evaluate_trained_model(model, loader, labels=POKEMON_TYPES):
    y_true, y_pred, names = predict_loader(model, loader)
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f} ({acc * 100:.2f}%)")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    plt.figure(figsize=(14, 12))
    if sns is not None:
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    else:
        plt.imshow(cm, cmap="Blues")
        plt.colorbar()
        plt.xticks(range(len(labels)), labels, rotation=45)
        plt.yticks(range(len(labels)), labels)
    plt.title(f"Matriz de confusion - accuracy {acc:.4f}")
    plt.xlabel("Predicho")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.show()

    return y_true, y_pred, names

# Ejemplo de uso luego de entrenar:
# best_model_path = OUTPUT_DIR / "inception_mixed_7c_best.pt"
# if best_model_path.exists():
#     model = build_inception_model(num_classes=num_types, fine_tune="mixed_7c")
#     model, checkpoint = load_checkpoint(model, best_model_path)
#     y_true, y_pred, names = evaluate_trained_model(model, val_loader)

In [ ]:
def show_prediction_examples(model, dataset, n=12):
    model.eval()
    indices = np.random.default_rng(SEED).choice(len(dataset), size=min(n, len(dataset)), replace=False)

    fig = plt.figure(figsize=(14, 8))
    with torch.no_grad():
        for plot_idx, dataset_idx in enumerate(indices):
            item = dataset[dataset_idx]
            image = item["image"].unsqueeze(0).to(device)
            output = model(image)
            pred_idx = int(output.argmax(dim=1).cpu().item())
            true_idx = int(item["label"].item())

            plt.subplot(3, 4, plot_idx + 1)
            plt.imshow(denormalize_image(item["image"]))
            title_color = "green" if pred_idx == true_idx else "red"
            plt.title(f"Real: {idx_to_type[true_idx]}\nPred: {idx_to_type[pred_idx]}", color=title_color, fontsize=10)
            plt.axis("off")

    plt.suptitle("Ejemplos cualitativos de prediccion", fontsize=16)
    plt.tight_layout()
    plt.show()

# Ejemplo de uso luego de cargar un checkpoint:
# show_prediction_examples(model, val_dataset, n=12)

### Vistas adicionales para analisis de errores

Estas funciones no entrenan el modelo: sirven para mirar con mas detalle que aprendio. Son importantes para el informe porque permiten pasar de una metrica global a preguntas concretas: que tipos se confunden, que clases tienen peor recall y si los errores son razonables visualmente.

In [ ]:
def predict_loader_with_probs(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    probs_all = []
    names = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            labels = batch["label"].cpu().numpy()
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)

            y_true.extend(labels.tolist())
            y_pred.extend(preds.tolist())
            probs_all.extend(probs.tolist())
            names.extend(batch.get("name", [""] * len(labels)))

    return np.array(y_true), np.array(y_pred), np.array(probs_all), names


def analyze_errors_by_class(y_true, y_pred, labels=POKEMON_TYPES):
    rows = []
    for idx, label in enumerate(labels):
        mask = y_true == idx
        total = int(mask.sum())
        correct = int((y_pred[mask] == idx).sum()) if total > 0 else 0
        recall = correct / total if total > 0 else 0.0
        wrong_preds = y_pred[mask & (y_pred != idx)]
        most_confused = "-"
        if len(wrong_preds) > 0:
            confused_idx = Counter(wrong_preds.tolist()).most_common(1)[0][0]
            most_confused = labels[confused_idx]
        rows.append({
            "type": label,
            "total": total,
            "correct": correct,
            "recall": recall,
            "most_confused_with": most_confused,
        })
    return pd.DataFrame(rows).sort_values("recall")


def show_error_examples(model, dataset, max_examples=12):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    y_true, y_pred, probs, names = predict_loader_with_probs(model, loader)
    error_indices = np.where(y_true != y_pred)[0]

    if len(error_indices) == 0:
        print("No hay errores para mostrar.")
        return

    selected = error_indices[:max_examples]
    fig = plt.figure(figsize=(14, 8))
    for plot_idx, dataset_idx in enumerate(selected):
        item = dataset[int(dataset_idx)]
        pred_idx = int(y_pred[dataset_idx])
        true_idx = int(y_true[dataset_idx])
        confidence = float(probs[dataset_idx, pred_idx])

        plt.subplot(3, 4, plot_idx + 1)
        plt.imshow(denormalize_image(item["image"]))
        plt.title(
            f"Real: {idx_to_type[true_idx]}\nPred: {idx_to_type[pred_idx]} ({confidence:.2f})",
            color="red",
            fontsize=9,
        )
        plt.axis("off")

    plt.suptitle("Ejemplos de errores del modelo", fontsize=16)
    plt.tight_layout()
    plt.show()


def show_topk_predictions(model, dataset, indices=None, k=3):
    if indices is None:
        indices = list(range(min(8, len(dataset))))

    model.eval()
    rows = []
    with torch.no_grad():
        for idx in indices:
            item = dataset[int(idx)]
            image = item["image"].unsqueeze(0).to(device)
            probs = torch.softmax(model(image), dim=1).cpu().numpy()[0]
            top_indices = probs.argsort()[-k:][::-1]
            row = {
                "name": item["name"],
                "true_type": idx_to_type[int(item["label"].item())],
            }
            for rank, pred_idx in enumerate(top_indices, start=1):
                row[f"top_{rank}"] = idx_to_type[int(pred_idx)]
                row[f"top_{rank}_prob"] = float(probs[pred_idx])
            rows.append(row)
    return pd.DataFrame(rows)

# Ejemplo de uso luego de cargar un checkpoint:
# y_true, y_pred, probs, names = predict_loader_with_probs(model, val_loader)
# display(analyze_errors_by_class(y_true, y_pred))
# show_error_examples(model, val_dataset, max_examples=12)
# display(show_topk_predictions(model, val_dataset, indices=[0, 1, 2, 3], k=3))

## 9. Experimentos propuestos para completar el informe

La consigna pide comparar parametros y discutir resultados. En el esqueleto de la catedra aparece esta misma logica: primero un modelo simple, despues graficas de entrenamiento, evaluacion con matriz de confusion y finalmente regularizacion con data augmentation, dropout y AdamW. Para este TP hacemos la version equivalente en PyTorch/InceptionV3.

La corrida `Mixed_7c` queda como baseline principal. A partir de ahi proponemos tres variantes controladas:

| Experimento | Capas entrenables | Regularizacion | Hipotesis | Que documentar |
|---|---|---|---|---|
| `exp_01_classifier_only` | Solo `fc` y `AuxLogits.fc` | Ninguna extra | Menos riesgo de sobreajuste, pero menor adaptacion al dominio Pokemon. | Si aprende menos que el baseline. |
| `exp_02_mixed_7c_dropout_adamw` | `Mixed_7c` + cabezas | Dropout + AdamW | Deberia bajar la brecha train/val respecto del baseline. | Comparar accuracy, macro-F1 y loss. |
| `exp_03_mixed_7b_7c_dropout_adamw` | `Mixed_7b`, `Mixed_7c` + cabezas | Dropout + AdamW | Mas adaptacion al dominio visual Pokemon. | Ver si mejora validacion o si sobreajusta. |
| `exp_04_balanced_loss_sampler` | `Mixed_7c` + cabezas | Dropout + AdamW + pesos de clase + sampler | Mejor desempeno sobre tipos minoritarios. | Comparar macro-F1 y reporte por clase. |

No conviene cambiar todo a la vez sin registrar nada. Cada experimento debe tener nombre, hipotesis, parametros, grafica, mejor epoca y una lectura cualitativa.

In [ ]:
experiment_configs = [
    {
        "run_name": "exp_01_classifier_only",
        "fine_tune": "classifier_only",
        "dropout": 0.0,
        "optimizer_name": "adam",
        "weight_decay": 0.0,
        "use_class_weights": False,
        "use_weighted_sampler": False,
        "epochs": 15,
        "hypothesis": "Entrenar solo la cabeza deberia sobreajustar menos, pero posiblemente obtenga menor accuracy.",
    },
    {
        "run_name": "exp_02_mixed_7c_dropout_adamw",
        "fine_tune": "mixed_7c",
        "dropout": 0.4,
        "optimizer_name": "adamw",
        "weight_decay": 1e-4,
        "use_class_weights": False,
        "use_weighted_sampler": False,
        "epochs": 15,
        "hypothesis": "Dropout y AdamW deberian reducir la brecha entre entrenamiento y validacion.",
    },
    {
        "run_name": "exp_03_mixed_7b_7c_dropout_adamw",
        "fine_tune": "mixed_7b_7c",
        "dropout": 0.4,
        "optimizer_name": "adamw",
        "weight_decay": 1e-4,
        "use_class_weights": False,
        "use_weighted_sampler": False,
        "epochs": 15,
        "hypothesis": "Descongelar Mixed_7b y Mixed_7c puede adaptar mejor el extractor al dominio Pokemon.",
    },
    {
        "run_name": "exp_04_balanced_loss_sampler",
        "fine_tune": "mixed_7c",
        "dropout": 0.4,
        "optimizer_name": "adamw",
        "weight_decay": 1e-4,
        "use_class_weights": True,
        "use_weighted_sampler": True,
        "epochs": 15,
        "hypothesis": "El balanceo deberia mejorar macro-F1 y clases minoritarias, aunque no necesariamente accuracy global.",
    },
]

pd.DataFrame(experiment_configs)

In [ ]:
def run_experiment(config):
    print(f"\n=== {config['run_name']} ===")
    print(config["hypothesis"])

    train_loader_exp = make_train_loader(use_weighted_sampler=config["use_weighted_sampler"])
    model_exp = build_inception_model(
        num_classes=num_types,
        fine_tune=config["fine_tune"],
        dropout=config["dropout"],
    ).to(device)

    total_params, trainable_params = count_parameters(model_exp)
    print(f"Parametros entrenables: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    criterion_exp = make_criterion(use_class_weights=config["use_class_weights"])
    optimizer_exp = make_optimizer(
        model_exp,
        fine_tune_lr=1e-4,
        head_lr=1e-3,
        weight_decay=config["weight_decay"],
        optimizer_name=config["optimizer_name"],
    )
    scheduler_exp = optim.lr_scheduler.StepLR(optimizer_exp, step_size=5, gamma=0.5)

    history_df, best_path = train_model(
        model_exp,
        train_loader_exp,
        val_loader,
        criterion_exp,
        optimizer_exp,
        scheduler=scheduler_exp,
        epochs=config["epochs"],
        run_name=config["run_name"],
        early_stopping_patience=5,
    )

    plot_history(history_df, title=config["run_name"])
    return history_df, best_path

# Recomendacion practica: ejecutar de a un experimento, no todos juntos.
# config = experiment_configs[1]
# history_df, best_model_path = run_experiment(config)

In [ ]:
def summarize_saved_histories(output_dir=OUTPUT_DIR):
    rows = []
    for history_path in sorted(Path(output_dir).glob("*_history.csv")):
        history = pd.read_csv(history_path)
        best_idx = history["val_acc"].idxmax()
        best_row = history.loc[best_idx]
        rows.append({
            "run_name": history_path.name.replace("_history.csv", ""),
            "best_epoch": int(best_row["epoch"]),
            "best_val_acc": best_row["val_acc"],
            "best_val_macro_f1": best_row.get("val_macro_f1", np.nan),
            "final_train_acc": history.iloc[-1]["train_acc"],
            "final_val_acc": history.iloc[-1]["val_acc"],
            "generalization_gap": history.iloc[-1]["train_acc"] - history.iloc[-1]["val_acc"],
        })
    return pd.DataFrame(rows).sort_values("best_val_acc", ascending=False) if rows else pd.DataFrame()

# Luego de ejecutar experimentos:
# results_summary = summarize_saved_histories()
# display(results_summary)

## 10. Como documentar las corridas

Para presentar el trabajo formalmente, cada corrida deberia quedar registrada con el mismo esquema:

1. **Objetivo de la corrida:** que se quiso probar y por que.
2. **Configuracion:** capas descongeladas, dropout, optimizador, weight decay, batch size, epocas, scheduler y si se uso balanceo.
3. **Hipotesis previa:** que esperabamos que ocurra antes de ver el resultado.
4. **Resultados cuantitativos:** mejor `val_acc`, mejor `val_macro_f1`, loss final y brecha entre train y validacion.
5. **Resultados cualitativos:** matriz de confusion y algunos ejemplos de aciertos/errores.
6. **Conclusion de la corrida:** si se acepta o rechaza la hipotesis y que decision motiva para el siguiente experimento.

La parte importante para un TP final no es solamente obtener el mayor accuracy, sino mostrar una secuencia razonada de decisiones. Si una variante no mejora, igual sirve: se documenta como evidencia de que cierto mecanismo no ayudo en este dataset.

## 11. Discusion preliminar

La primera corrida confirma que transfer learning es una estrategia razonable para el problema. El modelo parte de representaciones visuales generales aprendidas en ImageNet y, con pocas capas entrenables, logra una validacion final de `58.46%`. Para 18 clases, y considerando que los tipos Pokemon no siempre son visualmente evidentes, el resultado es aceptable como baseline inicial.

La principal limitacion observada es la brecha entre entrenamiento y validacion. El entrenamiento llega a `86.99%`, pero validacion queda en `58.46%`. Esto puede deberse a tamano reducido del dataset, clases desbalanceadas, etiquetas con ambiguedad semantica y disenos de Pokemon que comparten rasgos entre tipos. Tambien hay una limitacion conceptual: usar solo `Type 1` ignora Pokemon de doble tipo, por lo que una imagen visualmente asociada a su tipo secundario puede ser penalizada como incorrecta.

Como trabajo futuro proponemos: evaluar metricas por clase, aplicar ponderacion por frecuencia de clase, probar `WeightedRandomSampler`, experimentar con mas o menos capas descongeladas, guardar checkpoints por corrida, incorporar Grad-CAM para interpretar regiones de la imagen usadas por la red y extender el problema a clasificacion multi-label usando `Type 1` y `Type 2`.

## 12. Borrador de conclusiones

En este trabajo se implemento un clasificador de tipo primario de Pokemon mediante fine-tuning de InceptionV3. La metodologia siguio la linea propuesta por la catedra: tomar un modelo moderno preentrenado, adaptarlo a una tarea nueva, medir resultados y discutir sus limites. El pipeline incluye carga desde Hugging Face, conversion de imagenes a RGB, normalizacion compatible con ImageNet, data augmentation, reemplazo de la capa final y entrenamiento diferenciado de las capas profundas.

La corrida registrada muestra aprendizaje sostenido durante las 15 epocas y alcanza `58.46%` de accuracy de validacion. Aunque el valor no es definitivo, permite defender que la transferencia de aprendizaje aporta una base util. La diferencia entre entrenamiento y validacion indica que el siguiente foco deberia ser controlar el sobreajuste y mirar con mas detalle los errores por clase. En particular, la matriz de confusion y el reporte de clasificacion son necesarios para explicar que tipos se reconocen mejor y cuales generan confusion.